# Notebook 01 — Prepare your manufacturing data

**What you’ll do:** Create tables and **sample manufacturing data** in Unity Catalog (plants, lines, production events, quality, safety). You also get a few SQL helper functions Genie can use.

**Why it matters:** Genie answers questions from these tables—this step is your foundation.

**Workshop order:** 01 → 02 → 03 → 04 → 05 → 06 → 07 → 08 → 09. Run **01** before **02**.

**Time:** About **2 minutes** on Serverless (**Run all**).

## Compute

Choose **Serverless** (recommended) or a Unity Catalog–enabled cluster. Set **catalog** and **schema** with the widgets below only—do not use `spark.conf.set()` for workshop settings.


## Your catalog and schema

Set the **Catalog** and **Schema** widgets to a location **you can write to**. If the catalog does not exist yet, create it in **Catalog Explorer** first (jobs often cannot create catalogs). The defaults are **examples**—change them for your environment.

**Important:** Use the **same** catalog and schema in **every** notebook (**02**–**09**) so Genie and `workshop_config` stay aligned.

After **Install dependencies**, Python restarts—re-run the config cell below if needed, using the same values.


In [ ]:
dbutils.widgets.text("catalog", "workshop_demo", "Catalog")
dbutils.widgets.text("schema", "genie_workshop_manufacturing", "Schema")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")

# Serverless / jobs: use an existing catalog you have CREATE SCHEMA on (no spark.conf.set).
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
print(f"Using {CATALOG}.{SCHEMA}")


## Install dependencies

On **serverless**, `%pip` installs into the notebook environment; `restartPython()` clears the interpreter so the next cells pick up new packages.


In [ ]:
%pip install databricks-sdk==0.74.0 faker --quiet

## Restart Python after pip install

In [ ]:
dbutils.library.restartPython()

## Re-set config variables after Python restart

Python restart clears all variables, so we re-declare them here.

In [ ]:
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
print(f"Using {CATALOG}.{SCHEMA}")


## Step 1: Create Unity Catalog schema

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
print(f"Using {CATALOG}.{SCHEMA}")

## Step 2: Generate synthetic manufacturing data

Creates 7 tables covering plants, production lines, operators, production events, quality metrics, safety incidents, and equipment feedback.

Each **`production_events`** row includes a synthetic **`unit_serial_vin`** (17-character traceability id) used in notebook **07** to demonstrate column masking in Genie.

In [ ]:
import random
import json
from datetime import datetime, timedelta, date
from faker import Faker
from pyspark.sql.types import *
from pyspark.sql import functions as F

fake = Faker()
Faker.seed(42)
random.seed(42)

PLANTS = [
    {"plant_id": "P001", "plant_name": "Midwest Assembly Plant",     "city": "Detroit",       "state": "Michigan",    "country": "USA", "annual_revenue": 2400000000.0, "employee_count": 4500},
    {"plant_id": "P002", "plant_name": "Great Lakes Stamping",       "city": "Flint",         "state": "Michigan",    "country": "USA", "annual_revenue": 1800000000.0, "employee_count": 3200},
    {"plant_id": "P003", "plant_name": "Southern Paint & Body",      "city": "Spring Hill",   "state": "Tennessee",   "country": "USA", "annual_revenue": 2100000000.0, "employee_count": 3800},
    {"plant_id": "P004", "plant_name": "EV Battery Innovation Center","city": "Lordstown",    "state": "Ohio",        "country": "USA", "annual_revenue": 950000000.0,  "employee_count": 1200},
    {"plant_id": "P005", "plant_name": "Heartland Powertrain",       "city": "Tonawanda",     "state": "New York",    "country": "USA", "annual_revenue": 1600000000.0, "employee_count": 2900},
    {"plant_id": "P006", "plant_name": "Delta Assembly",             "city": "Lansing",       "state": "Michigan",    "country": "USA", "annual_revenue": 2200000000.0, "employee_count": 4100},
    {"plant_id": "P007", "plant_name": "Lone Star Truck Assembly",   "city": "Arlington",     "state": "Texas",       "country": "USA", "annual_revenue": 3100000000.0, "employee_count": 5200},
    {"plant_id": "P008", "plant_name": "Pacific Coast Paint Facility","city": "Fremont",      "state": "California",  "country": "USA", "annual_revenue": 1400000000.0, "employee_count": 2200},
]

PRODUCT_TYPES = ["Sedan", "Truck", "SUV", "EV Battery Pack", "Crossover"]
LINE_TYPES = ["Assembly", "Paint", "Stamping", "EV Battery", "Powertrain"]

lines = []
line_counter = 1
for plant in PLANTS:
    n_lines = random.randint(3, 5)
    for i in range(n_lines):
        lt = random.choice(LINE_TYPES)
        pt = random.choice(PRODUCT_TYPES)
        lines.append({
            "line_id": f"L{line_counter:03d}",
            "line_name": f"{plant['plant_name'].split()[0]} {lt} Line {i+1}",
            "description": f"{lt} line producing {pt} components",
            "product_type": pt,
            "plant_id": plant["plant_id"],
            "daily_capacity": random.randint(200, 800),
            "cost_per_unit": round(random.uniform(50, 500), 2),
            "start_date": str(fake.date_between(start_date="-5y", end_date="-1y")),
            "end_date": None,
            "status": random.choice(["Active", "Active", "Active", "Maintenance"]),
        })
        line_counter += 1

SHIFTS = ["Morning", "Afternoon", "Night"]
CERTIFICATIONS = ["Welding", "Robotics", "Paint", "Quality", "Safety", "EV Systems", "Stamping"]
operators = []
for i in range(1, 501):
    operators.append({
        "operator_id": f"OP{i:04d}",
        "first_name": fake.first_name(),
        "last_name": fake.last_name(),
        "shift": random.choice(SHIFTS),
        "certification": random.choice(CERTIFICATIONS),
        "hire_date": str(fake.date_between(start_date="-15y", end_date="-6m")),
        "plant_id": random.choice([p["plant_id"] for p in PLANTS]),
    })

EVENT_TYPES = ["unit_produced", "unit_produced", "unit_produced", "unit_produced",
               "defect_detected", "inspection_passed", "inspection_passed",
               "downtime_start", "scrap", "rework_completed"]
DEFECT_CODES = [
    "DEF-WELD-001", "DEF-WELD-002", "DEF-WELD-003",
    "DEF-PAINT-001", "DEF-PAINT-002", "DEF-PAINT-003",
    "DEF-FIT-001", "DEF-FIT-002",
    "DEF-ELEC-001", "DEF-ELEC-002",
    "DEF-STMP-001", "DEF-STMP-002",
]

events = []
start_dt = date(2024, 1, 1)
end_dt = date(2024, 12, 31)
delta_days = (end_dt - start_dt).days

_VIN_CHARS = "ABCDEFGHJKLMNPRSTUVWXYZ0123456789"  # VIN alphabet (no I, O, Q)

def _fake_unit_serial_vin():
    return "".join(random.choices(_VIN_CHARS, k=17))

for i in range(1, 50001):
    et = random.choice(EVENT_TYPES)
    d = start_dt + timedelta(days=random.randint(0, delta_days))
    hour = random.randint(0, 23)
    minute = random.randint(0, 59)
    ts = datetime(d.year, d.month, d.day, hour, minute)
    line = random.choice(lines)
    op = random.choice(operators)
    events.append({
        "event_id": f"EVT{i:06d}",
        "event_type": et,
        "event_date": str(d),
        "event_timestamp": ts.isoformat(),
        "production_line_id": line["line_id"],
        "operator_id": op["operator_id"],
        "part_number": f"PN-{random.randint(10000, 99999)}",
        "unit_serial_vin": _fake_unit_serial_vin(),
        "defect_code": random.choice(DEFECT_CODES) if et == "defect_detected" else None,
    })

quality_metrics = []
for day_offset in range(delta_days + 1):
    d = start_dt + timedelta(days=day_offset)
    sampled_lines = random.sample(lines, min(6, len(lines)))
    for line in sampled_lines:
        units = random.randint(100, 500)
        defects = int(units * random.uniform(0.005, 0.08))
        quality_metrics.append({
            "date": str(d),
            "plant_id": line["plant_id"],
            "production_line_id": line["line_id"],
            "units_produced": units,
            "units_passed_inspection": units - defects,
            "defects_found": defects,
            "downtime_minutes": round(random.uniform(0, 120), 1),
            "scrap_count": random.randint(0, max(1, defects // 2)),
            "rework_count": random.randint(0, max(1, defects)),
            "first_pass_yield": round((units - defects) / units, 4),
            "oee_score": round(random.uniform(0.60, 0.98), 4),
        })

SEVERITY = ["Low", "Medium", "High", "Critical"]
CATEGORIES = ["Ergonomic", "Chemical Exposure", "Equipment Malfunction", "Slip/Trip/Fall", "Electrical", "Fire Hazard"]
RESOLUTIONS = ["Resolved", "Under Investigation", "Pending Action", "Closed"]
safety = []
for i in range(1, 41):
    line = random.choice(lines)
    safety.append({
        "incident_id": f"SI{i:04d}",
        "description": fake.sentence(nb_words=10),
        "severity": random.choice(SEVERITY),
        "production_line_id": line["line_id"],
        "incident_date": str(fake.date_between(start_date=start_dt, end_date=end_dt)),
        "resolution_status": random.choice(RESOLUTIONS),
        "category": random.choice(CATEGORIES),
        "root_cause": fake.sentence(nb_words=8),
        "corrective_action": fake.sentence(nb_words=8),
    })

EQUIPMENT_NAMES = ["Fanuc Robotic Welder", "KUKA Paint Robot", "Schuler Press", "ABB Assembly Arm",
                   "Siemens Conveyor", "Tesla Battery Stacker", "Atlas Copco Torque Gun", "Keyence Vision System"]
COMMENTS = [
    "Runs smoothly, no issues this week.",
    "Calibration drifting — needs maintenance soon.",
    "Excellent precision on welds today.",
    "Intermittent faults during second shift.",
    "Paint coverage has been inconsistent lately.",
    "Battery stacker alignment is perfect after recalibration.",
    "Torque gun readings are within spec.",
    "Conveyor belt slipping under heavy load.",
    "Vision system catching defects we used to miss.",
    "Needs replacement — frequent breakdowns.",
    "Outstanding performance after firmware update.",
    "Hydraulic leak detected — reported to maintenance.",
]
feedback = []
for i in range(1, 101):
    line = random.choice(lines)
    op = random.choice(operators)
    feedback.append({
        "feedback_id": f"FB{i:04d}",
        "equipment_name": random.choice(EQUIPMENT_NAMES),
        "comment": random.choice(COMMENTS),
        "feedback_date": str(fake.date_between(start_date=start_dt, end_date=end_dt)),
        "production_line_id": line["line_id"],
        "operator_id": op["operator_id"],
    })

print(f"Generated: {len(PLANTS)} plants, {len(lines)} lines, {len(operators)} operators, "
      f"{len(events)} events, {len(quality_metrics)} quality metrics, {len(safety)} safety incidents, {len(feedback)} feedback records")

## Step 3: Write data to Delta tables

Drops existing tables (if any) and writes each dataset as a managed Delta table.

In [ ]:
from pyspark.sql.types import *
from pyspark.sql import functions as F
import pandas as pd

fqn = f"{CATALOG}.{SCHEMA}"

schemas = {
    "plants": StructType([
        StructField("plant_id", StringType()), StructField("plant_name", StringType()),
        StructField("city", StringType()), StructField("state", StringType()),
        StructField("country", StringType()), StructField("annual_revenue", DoubleType()),
        StructField("employee_count", LongType()),
    ]),
    "production_lines": StructType([
        StructField("line_id", StringType()), StructField("line_name", StringType()),
        StructField("description", StringType()), StructField("product_type", StringType()),
        StructField("plant_id", StringType()), StructField("daily_capacity", LongType()),
        StructField("cost_per_unit", DoubleType()), StructField("start_date", StringType()),
        StructField("end_date", StringType(), True), StructField("status", StringType()),
    ]),
    "operators": StructType([
        StructField("operator_id", StringType()), StructField("first_name", StringType()),
        StructField("last_name", StringType()), StructField("shift", StringType()),
        StructField("certification", StringType()), StructField("hire_date", StringType()),
        StructField("plant_id", StringType()),
    ]),
    "production_events": StructType([
        StructField("event_id", StringType()), StructField("event_type", StringType()),
        StructField("event_date", StringType()), StructField("event_timestamp", StringType()),
        StructField("production_line_id", StringType()), StructField("operator_id", StringType()),
        StructField("part_number", StringType()), StructField("unit_serial_vin", StringType()),
        StructField("defect_code", StringType(), True),
    ]),
    "quality_metrics_daily": StructType([
        StructField("date", StringType()), StructField("plant_id", StringType()),
        StructField("production_line_id", StringType()), StructField("units_produced", LongType()),
        StructField("units_passed_inspection", LongType()), StructField("defects_found", LongType()),
        StructField("downtime_minutes", DoubleType()), StructField("scrap_count", LongType()),
        StructField("rework_count", LongType()), StructField("first_pass_yield", DoubleType()),
        StructField("oee_score", DoubleType()),
    ]),
    "safety_incidents": StructType([
        StructField("incident_id", StringType()), StructField("description", StringType()),
        StructField("severity", StringType()), StructField("production_line_id", StringType()),
        StructField("incident_date", StringType()), StructField("resolution_status", StringType()),
        StructField("category", StringType()), StructField("root_cause", StringType()),
        StructField("corrective_action", StringType()),
    ]),
    "equipment_feedback": StructType([
        StructField("feedback_id", StringType()), StructField("equipment_name", StringType()),
        StructField("comment", StringType()), StructField("feedback_date", StringType()),
        StructField("production_line_id", StringType()), StructField("operator_id", StringType()),
    ]),
}

tables_data = {
    "plants": PLANTS,
    "production_lines": lines,
    "operators": operators,
    "production_events": events,
    "quality_metrics_daily": quality_metrics,
    "safety_incidents": safety,
    "equipment_feedback": feedback,
}

for table_name, data in tables_data.items():
    spark.sql(f"DROP TABLE IF EXISTS {fqn}.{table_name}")
    pdf = pd.DataFrame(data)
    df = spark.createDataFrame(pdf, schema=schemas[table_name])
    df.write.mode("overwrite").saveAsTable(f"{fqn}.{table_name}")
    count = spark.table(f"{fqn}.{table_name}").count()
    print(f"✓ {table_name}: {count} rows")

## Step 4: Optimize column types

Cast string date columns to proper DATE types for correct filtering and dashboard behavior.

In [ ]:
fqn = f"{CATALOG}.{SCHEMA}"

spark.sql(f"""
CREATE OR REPLACE TABLE {fqn}.production_events AS
SELECT event_id, event_type, CAST(event_date AS DATE) AS event_date,
       event_timestamp, production_line_id, operator_id, part_number, unit_serial_vin, defect_code
FROM {fqn}.production_events
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {fqn}.production_lines AS
SELECT line_id, line_name, description, product_type, plant_id,
       daily_capacity, cost_per_unit,
       CAST(start_date AS DATE) AS start_date,
       CAST(end_date AS DATE) AS end_date, status
FROM {fqn}.production_lines
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {fqn}.quality_metrics_daily AS
SELECT CAST(date AS DATE) AS date, plant_id, production_line_id,
       units_produced, units_passed_inspection, defects_found,
       downtime_minutes, scrap_count, rework_count,
       first_pass_yield, oee_score
FROM {fqn}.quality_metrics_daily
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {fqn}.safety_incidents AS
SELECT incident_id, description, severity, production_line_id,
       CAST(incident_date AS DATE) AS incident_date,
       resolution_status, category, root_cause, corrective_action
FROM {fqn}.safety_incidents
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {fqn}.operators AS
SELECT operator_id, first_name, last_name, shift, certification,
       CAST(hire_date AS DATE) AS hire_date, plant_id
FROM {fqn}.operators
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {fqn}.equipment_feedback AS
SELECT feedback_id, equipment_name, comment,
       CAST(feedback_date AS DATE) AS feedback_date,
       production_line_id, operator_id
FROM {fqn}.equipment_feedback
""")

print("✓ All date columns cast to DATE type")

## Step 5: Create analytics functions

Build three reusable SQL table-valued functions for defect rate analysis, OEE computation, and best production line lookup. These functions are available to Genie for natural language queries.

In [ ]:
fqn = f"{CATALOG}.{SCHEMA}"

spark.sql(f"""
CREATE OR REPLACE FUNCTION {fqn}.compute_defect_rate(
  line_id STRING COMMENT 'Production line ID, e.g. L001',
  start_dt DATE  COMMENT 'Start date, e.g. 2024-01-01',
  end_dt DATE    COMMENT 'End date, e.g. 2024-06-30'
)
RETURNS TABLE (
  production_line STRING,
  total_units INT,
  total_defects INT,
  defect_rate DOUBLE
)
COMMENT 'Returns defect rate for a production line over a date range'
RETURN
  SELECT
    pl.line_name AS production_line,
    COUNT(CASE WHEN pe.event_type = 'unit_produced' THEN 1 END) AS total_units,
    COUNT(CASE WHEN pe.event_type = 'defect_detected' THEN 1 END) AS total_defects,
    ROUND(COUNT(CASE WHEN pe.event_type = 'defect_detected' THEN 1 END) * 100.0 /
      NULLIF(COUNT(CASE WHEN pe.event_type = 'unit_produced' THEN 1 END), 0), 2) AS defect_rate
  FROM {fqn}.production_events pe
  JOIN {fqn}.production_lines pl ON pe.production_line_id = pl.line_id
  WHERE pe.production_line_id = line_id
    AND pe.event_date BETWEEN start_dt AND end_dt
  GROUP BY pl.line_name
""")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {fqn}.compute_oee_summary(
  p_id STRING  COMMENT 'Plant ID, e.g. P001',
  start_dt DATE COMMENT 'Start date, e.g. 2024-01-01',
  end_dt DATE   COMMENT 'End date, e.g. 2024-06-30'
)
RETURNS TABLE (
  plant_name STRING,
  avg_oee DOUBLE,
  avg_first_pass_yield DOUBLE,
  total_downtime_hours DOUBLE
)
COMMENT 'Returns OEE summary for a plant over a date range'
RETURN
  SELECT
    p.plant_name,
    ROUND(AVG(qm.oee_score) * 100, 2) AS avg_oee,
    ROUND(AVG(qm.first_pass_yield) * 100, 2) AS avg_first_pass_yield,
    ROUND(SUM(qm.downtime_minutes) / 60.0, 1) AS total_downtime_hours
  FROM {fqn}.quality_metrics_daily qm
  JOIN {fqn}.plants p ON qm.plant_id = p.plant_id
  WHERE qm.plant_id = p_id
    AND qm.date BETWEEN start_dt AND end_dt
  GROUP BY p.plant_name
""")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {fqn}.get_best_production_line()
RETURNS TABLE (
  production_line STRING,
  product_type STRING,
  avg_first_pass_yield DOUBLE,
  total_units_produced INT
)
COMMENT 'Returns the production line with the highest first pass yield'
RETURN
  SELECT
    pl.line_name AS production_line,
    pl.product_type,
    ROUND(AVG(qm.first_pass_yield) * 100, 2) AS avg_first_pass_yield,
    SUM(qm.units_produced) AS total_units_produced
  FROM {fqn}.quality_metrics_daily qm
  JOIN {fqn}.production_lines pl ON qm.production_line_id = pl.line_id
  GROUP BY pl.line_name, pl.product_type
  ORDER BY avg_first_pass_yield DESC
  LIMIT 1
""")

print("✓ Created functions: compute_defect_rate, compute_oee_summary, get_best_production_line")

## Step 6: Table comments for Genie

Short **table-level** comments help Genie and analysts understand grain and domain. Adjust text if you extend the model.

In [ ]:
fqn = f"{CATALOG}.{SCHEMA}"
comments = {
    "plants": "Manufacturing sites with capacity and revenue context.",
    "production_lines": "Assembly/paint/stamping/EV lines; join to plants and events.",
    "operators": "Shop-floor operators, shift, certifications, home plant.",
    "production_events": "Event grain: unit_produced, defect_detected, scrap, downtime, rework, inspection. unit_serial_vin is a 17-char unit traceability id (sensitive).",
    "quality_metrics_daily": "Daily OEE (0-1), first_pass_yield (0-1), downtime_minutes per line.",
    "safety_incidents": "Safety events tied to production_line_id with severity.",
    "equipment_feedback": "Operator comments on equipment by line.",
}
for t, cmt in comments.items():
    esc = cmt.replace("'", "''")
    spark.sql(f"ALTER TABLE {fqn}.{t} SET TBLPROPERTIES ('comment' = '{esc}')")
print("Applied table comments.")


## Setup complete

You now have **7 Delta tables** and **3 SQL functions** (`compute_defect_rate`, `compute_oee_summary`, `get_best_production_line`).

**Next:** run `02_create_genie_spaces` to create the **blank** and **configured** Genie spaces and save their IDs to `workshop_config`.